In [101]:
import pandas as pd
import numpy as np

In [102]:
weighted_returns_df = pd.read_csv("data/outputs/baseline_portfolio",
                                  index_col = 0, parse_dates = [1])
daily_df_annually = pd.read_csv("data/outputs/annually_reblanced_portfolio",
                                index_col = 0, parse_dates = [1])
daily_df_quarterly = pd.read_csv("data/outputs/quarterly_reblanced_portfolio",
                                 index_col = 0, parse_dates = [1])
daily_df_quarterly_band = pd.read_csv("data/outputs/quarterly_reblanced_byband_portfolio", 
                                      index_col = 0, parse_dates = [1])
annual_weights_frame = pd.read_csv("data/outputs/annually_rebalanced_weights",
                                index_col = 0, parse_dates=[1])
quarterly_weights_frame= pd.read_csv("data/outputs/quarterly_rebalanced_weights",
                                index_col = 0)
band_weights_frame = pd.read_csv("data/outputs/quarterly_rebalanced_byband_weights",
                                index_col = 0)

In [111]:
annual_weights_frame

,Date,weight_type,AGG,EEM,EFA,GLD,IWM,LQD,MTUM,QQQ,QUAL,SPY,TLT,USMV,VLUE,VNQ
0,2022-01-01,starting,0.150000,0.000000e+00,1.500000e-01,0.000000,0.000000e+00,0.000000,0.000000e+00,5.000000e-02,0.000000e+00,0.150000,1.500000e-01,0.150000,0.050000,1.500000e-01
1,2022-12-31,ending,0.161494,0.000000e+00,1.589608e-01,0.000000,0.000000e+00,0.000000,0.000000e+00,4.172837e-02,0.000000e+00,0.151925,1.276777e-01,0.168163,0.053120,1.369304e-01
2,2023-01-01,starting,0.150000,0.000000e+00,1.500000e-01,0.150000,1.500000e-01,0.150000,0.000000e+00,1.488532e-17,0.000000e+00,0.050000,0.000000e+00,0.086249,0.113751,0.000000e+00
3,2023-12-31,ending,0.139924,0.000000e+00,1.567541e-01,0.149243,1.547203e-01,0.144889,0.000000e+00,2.035145e-17,0.000000e+00,0.055700,0.000000e+00,0.084013,0.114756,0.000000e+00
4,2024-01-01,starting,0.150000,2.884593e-17,1.500000e-01,0.150000,8.939776e-17,0.150000,2.259489e-17,1.500000e-01,1.500000e-01,0.050000,5.181237e-16,0.050000,0.000000,4.465968e-17
5,2024-12-31,ending,0.133234,2.693197e-17,1.360986e-01,0.166569,8.730215e-17,0.132641,2.632492e-17,1.651507e-01,1.608208e-01,0.054747,4.176816e-16,0.050739,0.000000,4.103932e-17
6,2025-01-01,starting,0.150000,1.500000e-01,0.000000e+00,0.150000,0.000000e+00,0.150000,1.500000e-01,5.000000e-02,0.000000e+00,0.150000,0.000000e+00,0.050000,0.000000,0.000000e+00
7,2025-12-31,ending,0.129341,1.616647e-01,0.000000e+00,0.197497,0.000000e+00,0.130190,1.473898e-01,4.857598e-02,0.000000e+00,0.142044,0.000000e+00,0.043298,0.000000,0.000000e+00
8,2026-01-01,starting,0.150000,1.500000e-01,5.308254e-16,0.150000,1.242124e-16,0.150000,0.000000e+00,5.000000e-02,1.257812e-16,0.150000,4.662246e-16,0.050000,0.150000,0.000000e+00
9,2026-07-08,ending,0.135132,1.633741e-01,5.222397e-16,0.127445,1.343649e-16,0.134766,0.000000e+00,5.216137e-02,1.243969e-16,0.148562,4.144888e-16,0.047243,0.191317,0.000000e+00


In [103]:
## Portfolio Drift ##

def drift_evaluation(df):
    starting_weights = df[df["weight_type"] == "starting"]
    ending_weights = df[df["weight_type"] == "ending"]

    difference = np.array(ending_weights.drop(columns=['Date','weight_type'])) - np.array(starting_weights.drop(columns=['Date','weight_type']))

    drift_list = []

    for period in range(len(difference)):
        start_date = starting_weights['Date'].iloc[period]
        end_date = ending_weights['Date'].iloc[period]

        drift = abs(difference[period]).sum()
        drift_list.append({
            "Start": start_date,
            "End" : end_date,
            "Drift": drift
            })

    return pd.DataFrame(drift_list)

In [104]:
## Trades ##
def calculate_trades(df):
    starting_weights = df[df["weight_type"] == "starting"]
    ending_weights = df[df["weight_type"] == "ending"]
    starting_weights_next = starting_weights.drop(starting_weights.index[0])
    ending_weights_current = ending_weights.drop(ending_weights.index[-1])
    start = starting_weights_next.drop(columns=['Date','weight_type'])
    end = ending_weights_current.drop(columns=['Date','weight_type'])
    trades = np.array(start) - np.array(end)
    trades_df = pd.DataFrame(trades, index = starting_weights_next['Date'], columns = start.columns)
    return trades_df


In [105]:

annual_trades_df = calculate_trades(annual_weights_frame)
annual_trades_df

,AGG,EEM,EFA,GLD,IWM,LQD,MTUM,QQQ,QUAL,SPY,TLT,USMV,VLUE,VNQ
Date,,,,,,,,,,,,,,
2023-01-01,-0.011494,0.000000e+00,-8.960803e-03,0.150000,1.500000e-01,0.150000,0.000000e+00,-0.041728,0.000000e+00,-0.101925,-1.276777e-01,-0.081914,0.060631,-1.369304e-01
2024-01-01,0.010076,2.884593e-17,-6.754119e-03,0.000757,-1.547203e-01,0.005111,2.259489e-17,0.150000,1.500000e-01,-0.005700,5.181237e-16,-0.034013,-0.114756,4.465968e-17
2025-01-01,0.016766,1.500000e-01,-1.360986e-01,-0.016569,-8.730215e-17,0.017359,1.500000e-01,-0.115151,-1.608208e-01,0.095253,-4.176816e-16,-0.000739,0.000000,-4.103932e-17
2026-01-01,0.020659,-1.166472e-02,5.308254e-16,-0.047497,1.242124e-16,0.019810,-1.473898e-01,0.001424,1.257812e-16,0.007956,4.662246e-16,0.006702,0.150000,0.000000e+00


In [106]:
## Turnover ## 
def calculate_turnover(df):
    turnover_list = []
    for row in range(len(df.index)):
        turnover = abs(df.iloc[row]).sum(axis = 0) * 0.5
        turnover_list.append({
            'Trade Date':df.index[row],
            'turnover': turnover
            })
    return pd.DataFrame(turnover_list)


In [107]:

annual_portfolio_turnover = calculate_turnover(annual_trades_df)
annual_portfolio_turnover

,Trade Date,turnover
0,2023-01-01,0.510631
1,2024-01-01,0.315944
2,2025-01-01,0.429378
3,2026-01-01,0.206552


In [108]:
def transaction_cost_analysis(df, turnover_df, bp):
    cost_list = []
    transaction_cost = bp / 10000

    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])

    trade_days = pd.to_datetime(turnover_df["Trade Date"])
    turnover = turnover_df["turnover"].tolist()

    if "portfolio_value" not in df.columns:
        raise KeyError("df must contain a 'portfolio_value' column")

    for trade_day, turnover_value in zip(trade_days, turnover):
        prior_rows = df.loc[df["Date"] < trade_day]
        if prior_rows.empty:
            raise ValueError(f"No prior portfolio value available before {trade_day}")

        prior_row = prior_rows.iloc[-1]
        portfolio_value = prior_row["portfolio_value"]
        cost = portfolio_value * turnover_value * transaction_cost

        cost_list.append({
            "reblance_day": trade_day,
            "cost": cost,
        })

    return pd.DataFrame(cost_list)


In [109]:

annually_rebalanced_tc = transaction_cost_analysis(
    df = daily_df_annually,
    turnover_df=annual_portfolio_turnover,
    bp = 10
)
annually_rebalanced_tc

,reblance_day,cost
0,2023-01-01,0.000414
1,2024-01-01,0.000291
2,2025-01-01,0.000451
3,2026-01-01,0.000269


In [110]:
day = annually_rebalanced_tc['reblance_day']
values = daily_df_annually.loc[daily_df_annually['Rebalanced'] == 1, 'Date']
values, day

(0      2022-01-03
 251    2023-01-03
 501    2024-01-02
 753    2025-01-02
 1003   2026-01-02
 Name: Date, dtype: datetime64[us],
 0   2023-01-01
 1   2024-01-01
 2   2025-01-01
 3   2026-01-01
 Name: reblance_day, dtype: datetime64[us])

## Weight Stability ## 
For each rebalance:

Compute turnover
T
Observe portfolio value before the rebalance
V
Assume transaction cost
c
Cost paid
Cost=V×T×c
New portfolio value
V
after
	​

=V−Cost

Holding Period Analysis

Questions

Average holding period

How often is each ETF traded?

Average position age

Number of consecutive quarters held